In [ ]:
import re
import random
import numpy as np
import pandas as pd
import scipy.sparse as sp

from scipy.sparse import hstack
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, roc_auc_score

In [ ]:
# Load Data

# Load real notes from test set
print("Loading real notes...")
with open("../Data/processed/test.txt", "r", encoding="utf-8") as f:
    real_texts = f.read().split("\n<|endoftext|>\n")
real_texts = [t.strip() for t in real_texts if t.strip()]
print(f"Loaded {len(real_texts)} real notes")

# Load synthetic notes
distilgpt2_df = pd.read_csv("synthetic_notes_distilgpt2.csv")
qwen_df = pd.read_csv("synthetic_notes_qwen.csv")
llama_df = pd.read_csv("synthetic_notes_llama.csv")

print(f"DistilGPT-2: {len(distilgpt2_df)}, Qwen: {len(qwen_df)}, LLaMA: {len(llama_df)}")

In [ ]:
# Feature engineering
SECTION_HEADERS = [
    "Chief Complaint", "History of Present Illness",
    "Past Medical History", "Physical Exam", "Social History",
    "Family History", "Pertinent Results", "Brief Hospital Course",
    "Discharge Medications", "Discharge Instructions",
    "Discharge Diagnosis", "Followup Instructions",
    "Medications on Admission", "Discharge Disposition"
]

LAB_PATTERNS = [
    r'\bWBC[\-\s]', r'\bRBC[\-\s]', r'\bHgb[\-\s]', r'\bHct[\-\s]',
    r'\bNa[\-\s]', r'\bK[\-\s]', r'\bCr[\-\s]', r'\bALT[\-\s]',
    r'\bAST[\-\s]', r'\bPlt[\-\s]', r'\bINR[\-\s]', r'\bGlucose[\-\s]'
]

CONNECTORS = [
    "however", "furthermore", "additionally", "moreover", "therefore",
    "consequently", "subsequently", "notably", "overall", "in conclusion",
    "importantly", "specifically"
]

AI_PHRASES = [
    "it is important to note", "it is worth noting", "plays a crucial role",
    "comprehensive", "ensure", "furthermore", "in summary",
    "moving forward", "optimal", "utilize", "leverage"
]

def extract_features(text):
    text = str(text)
    sentences = [s.strip() for s in re.split(r'[.!?]', text) if s.strip()]
    sent_lengths = [len(s.split()) for s in sentences]
    words = text.split()
    lines = text.splitlines()
    n_words = len(words) if words else 1  # avoid div by zero
    n_chars = len(text) if text else 1

    f = {}
    # Linguistic 
    f["word_count"] = len(words)
    f["sentence_count"] = len(sentences)
    f["avg_sentence_length"] = np.mean(sent_lengths) if sent_lengths else 0
    f["vocab_richness"] = len(set(w.lower() for w in words)) / n_words
    f["num_lines"] = len(lines)
    f["avg_line_length"] = np.mean([len(l) for l in lines]) if lines else 0
    f["uppercase_word_ratio"] = sum(1 for w in words if w.isupper()) / n_words

    # Burstiness / sentence variance 
    f["sentence_length_std"] = np.std(sent_lengths) if len(sent_lengths) > 1 else 0
    f["sentence_length_cv"] = (f["sentence_length_std"] / f["avg_sentence_length"]
                                if f["avg_sentence_length"] > 0 else 0)

    # Readability
    syllable_est = sum(max(1, len(re.findall(r'[aeiouyAEIOUY]+', w))) for w in words)
    if sentences and words:
        f["flesch_reading_ease"] = (206.835 - 1.015 * (n_words / len(sentences))
                                     - 84.6 * (syllable_est / n_words))
    else:
        f["flesch_reading_ease"] = 0

    # Punctuation distribution 
    f["comma_density"] = text.count(",") / n_chars * 1000
    f["colon_density"] = text.count(":") / n_chars * 1000
    f["semicolon_density"] = text.count(";") / n_chars * 1000
    f["paren_density"] = text.count("(") / n_chars * 1000
    f["dash_density"] = text.count("-") / n_chars * 1000

    # Connector and AI-phrase density 
    text_lower = text.lower()
    f["connector_density"] = sum(text_lower.count(c) for c in CONNECTORS) / n_words * 100
    f["ai_phrase_density"] = sum(text_lower.count(p) for p in AI_PHRASES) / n_words * 100

    # Clinical / structural 
    f["phi_count"] = text.count("<PHI>")
    f["phi_density"] = f["phi_count"] / n_words
    f["section_header_count"] = sum(1 for h in SECTION_HEADERS if h.lower() in text_lower)
    f["lab_pattern_count"] = sum(1 for p in LAB_PATTERNS if re.search(p, text))
    f["has_markdown_bold"] = int(bool(re.search(r'\*\*', text)))
    f["has_markdown_headers"] = int(bool(re.search(r'###', text)))
    f["dash_list_count"] = len(re.findall(r'^\s*[-–]\s', text, re.MULTILINE))
    f["numbered_list_count"] = len(re.findall(r'^\s*\d+\.', text, re.MULTILINE))
    f["medication_keywords"] = len(re.findall(
        r'\b(mg|mcg|mL|tablet|capsule|daily|BID|TID|QID|PRN|PO|IV|SC)\b', text, re.IGNORECASE))
    f["vital_sign_patterns"] = len(re.findall(
        r'\b(BP|HR|RR|Temp|SpO2|O2|mmHg|bpm)\b', text, re.IGNORECASE))
    f["date_pattern_count"] = len(re.findall(r'\d{1,2}/\d{1,2}/\d{2,4}', text))

    return f

In [ ]:
# Building a balanced dataset
def build_dataset(real_texts, synthetic_df, n_real=1000, seed=42):
    random.seed(seed)
    sampled_real = random.sample(real_texts, n_real)
    texts = sampled_real + synthetic_df["generated_text"].astype(str).tolist()
    labels = [0] * n_real + [1] * len(synthetic_df)
    return texts, labels

distilgpt2_texts, distilgpt2_labels = build_dataset(real_texts, distilgpt2_df)
qwen_texts, qwen_labels = build_dataset(real_texts, qwen_df)
llama_texts, llama_labels = build_dataset(real_texts, llama_df)

In [ ]:
# Split datasets and fit TF-IDF on train test
def prepare_data(texts, labels):
    # Split on raw texts first to avoid TF-IDF leakage from test into train
    X_train_texts, X_test_texts, y_train, y_test = train_test_split(
        texts, labels, test_size=0.2, random_state=42, stratify=labels
    )

    tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)
    tfidf_train = tfidf.fit_transform(X_train_texts)
    tfidf_test = tfidf.transform(X_test_texts)

    hand_train = pd.DataFrame([extract_features(t) for t in X_train_texts])
    hand_test = pd.DataFrame([extract_features(t) for t in X_test_texts])
    feature_names = tfidf.get_feature_names_out().tolist() + hand_train.columns.tolist()

    X_train = hstack([tfidf_train, sp.csr_matrix(hand_train.values)])
    X_test = hstack([tfidf_test, sp.csr_matrix(hand_test.values)])

    return X_train, X_test, y_train, y_test, feature_names, tfidf

data_distilgpt2 = prepare_data(distilgpt2_texts, distilgpt2_labels)
data_qwen = prepare_data(qwen_texts, qwen_labels)
data_llama = prepare_data(llama_texts, llama_labels)
print("Feature matrices ready")

In [ ]:
# Train and evaluate
def train_classifier(data, name):
    X_train, X_test, y_train, y_test, feature_names, tfidf = data

    clf = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        eval_metric="logloss",
        random_state=42
    )
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]

    print(f"\n{'='*60}")
    print(f"Classifier: Real vs {name}")
    print(f"{'='*60}")
    print(classification_report(y_test, y_pred, target_names=["Real", "Synthetic"]))
    print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")

    return clf

clf_distilgpt2 = train_classifier(data_distilgpt2, "DistilGPT-2")
clf_qwen = train_classifier(data_qwen, "Qwen")
clf_llama = train_classifier(data_llama, "LLaMA")